In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.types import IntegerType, DecimalType 

# ============================================================ 

# BRONZE -> SILVER TRANSFORMATION 

# ============================================================ 

# Source and Target tables 

SOURCE_TABLE = "ujjivan_2.bronze.bank_transaction_fraud_detection" 

TARGET_TABLE = "ujjivan_2.silver.bank_transaction_fraud_detection" 

# ============================================================ 

# 1. READ BRONZE DATA 

# ============================================================ 
bronze_df = spark.table(SOURCE_TABLE) 

print("Bronze record count:", bronze_df.count()) 
# ============================================================ 

# 2. CLEAN AND TRANSFORM DATA 

# ============================================================ 
silver_df = ( 

    bronze_df 

    # ------------------------- 

    # Customer Information 

    # ------------------------- 

 
 

    .withColumn( 

        "Customer_ID", 

        F.trim(F.col("Customer_ID")) 

    ) 

    .withColumn( 

        "Customer_Name", 

        F.trim(F.col("Customer_Name")) 

    ) 
    .withColumn( 

        "Gender", 

        F.initcap(F.trim(F.col("Gender"))) 

    ) 
    .withColumn( 

        "Age", 

        F.col("Age").cast(IntegerType()) 

    ) 

    .withColumn( 

        "State", 

        F.initcap(F.trim(F.col("State"))) 

    ) 
    .withColumn( 

        "City", 

        F.initcap(F.trim(F.col("City"))) 

    ) 

    # ------------------------- 

    # Branch / Account 

    # ------------------------- 
    .withColumn( 

        "Bank_Branch", 

        F.trim(F.col("Bank_Branch")) 

    ) 

    .withColumn( 

        "Account_Type", 

        F.initcap(F.trim(F.col("Account_Type"))) 

    ) 

    # ------------------------- 

    # Transaction Information 

    # ------------------------- 
    .withColumn( 

        "Transaction_ID", 

        F.trim(F.col("Transaction_ID")) 

    ) 
    # Transaction Date 

    # Actual Bronze format: 

    # 2025-01-23 

    .withColumn( 

        "Transaction_Date", 

        F.to_date( 

            F.col("Transaction_Date").cast("string"), 

            "yyyy-MM-dd" 

        ) 

    ) 
    # Transaction Time 

    # Output format: 

    # HH:mm:ss 

    .withColumn( 

        "Transaction_Time", 

        F.date_format( 

            F.to_timestamp( 

                F.col("Transaction_Time").cast("string") 

            ), 

            "HH:mm:ss" 

        ) 

    ) 
    .withColumn( 

        "Transaction_Amount", 

        F.col("Transaction_Amount") 

         .cast(DecimalType(18, 2)) 

    ) 
    # ------------------------- 

    # Merchant Information 

    # ------------------------- 
    .withColumn( 

        "Merchant_ID", 

        F.trim(F.col("Merchant_ID")) 

    ) 
    .withColumn( 

        "Transaction_Type", 

        F.initcap(F.trim(F.col("Transaction_Type"))) 

    ) 
    .withColumn( 

        "Merchant_Category", 

        F.initcap(F.trim(F.col("Merchant_Category"))) 

    ) 
    # ------------------------- 

    # Account Balance 

    # ------------------------- 
    .withColumn( 

        "Account_Balance", 

        F.col("Account_Balance") 

         .cast(DecimalType(18, 2)) 

    ) 
    # ------------------------- 

    # Device Information 

    # ------------------------- 
    .withColumn( 

        "Transaction_Device", 

        F.initcap(F.trim(F.col("Transaction_Device"))) 

    ) 
    .withColumn( 

        "Transaction_Location", 

        F.trim(F.col("Transaction_Location")) 

    ) 
    .withColumn( 

        "Device_Type", 

        F.upper(F.trim(F.col("Device_Type"))) 

    ) 

    # ------------------------- 

    # Fraud 

    # ------------------------- 
    .withColumn( 

        "Is_Fraud", 

        F.col("Is_Fraud").cast(IntegerType()) 

    ) 
    # ------------------------- 

    # Currency 

    # ------------------------- 
    .withColumn( 

        "Transaction_Currency", 

        F.upper(F.trim(F.col("Transaction_Currency"))) 

    ) 
    # ------------------------- 

    # Customer Contact 

    # ------------------------- 
    .withColumn( 

        "Customer_Contact", 

        F.trim(F.col("Customer_Contact")) 

    ) 

    # ------------------------- 

    # Description 

    # ------------------------- 
    .withColumn( 

        "Transaction_Description", 

        F.trim(F.col("Transaction_Description")) 

    ) 

    # ------------------------- 

    # Email 

    # ------------------------- 

    .withColumn( 

        "Customer_Email", 

        F.lower(F.trim(F.col("Customer_Email"))) 

    ) 

) 
# ============================================================ 

# 3. CREATE DISTRICT COLUMN 

# ============================================================ 

# 

# Example: 

# 

# Thiruvananthapuram, Kerala 

#                         | 

#                         +--> Kerala 

# 

# Chennai, Tamil Nadu 

#                   | 

#                   +--> Tamil Nadu 

# 

# ============================================================ 

 
 

silver_df = silver_df.withColumn( 

    "District", 

    F.when( 

        F.col("Transaction_Location").contains(","), 

        F.trim( 

            F.element_at( 

                F.split( 

                    F.col("Transaction_Location"), 

                    "," 

                ), 

                -1 

            ) 

        ) 

    ).otherwise(None) 

) 

 
 
 

# ============================================================ 

# 4. DATA QUALITY FILTER 

# ============================================================ 

 
 

silver_df = silver_df.filter( 

    F.col("Customer_ID").isNotNull() 

    & 

    F.col("Transaction_ID").isNotNull() 

    & 

    F.col("Transaction_Date").isNotNull() 

) 

 
 
 

# ============================================================ 

# 5. REMOVE DUPLICATE TRANSACTIONS 

# ============================================================ 

 
 

silver_df = silver_df.dropDuplicates( 

    ["Transaction_ID"] 

) 

 
 
 

# ============================================================ 

# 6. ADD AUDIT COLUMNS 

# ============================================================ 

 
 

silver_df = ( 

    silver_df 

 
 

    .withColumn( 

        "_silver_processed_timestamp", 

        F.current_timestamp() 

    ) 

 
 

    .withColumn( 

        "_source_layer", 

        F.lit("bronze") 

    ) 

) 

 
 
 

# ============================================================ 

# 7. DISPLAY FINAL DATA 

# ============================================================ 

 
 

display( 

    silver_df.select( 

        "Customer_ID", 

        "Customer_Name", 

        "Gender", 

        "Age", 

        "State", 

        "City", 

        "Bank_Branch", 

        "Account_Type", 

        "Transaction_ID", 

        "Transaction_Date", 

        "Transaction_Time", 

        "Transaction_Amount", 

        "Merchant_ID", 

        "Transaction_Type", 

        "Merchant_Category", 

        "Account_Balance", 

        "Transaction_Device", 

        "Transaction_Location", 

        "District", 

        "Device_Type", 

        "Is_Fraud", 

        "Transaction_Currency", 

        "Customer_Contact", 

        "Transaction_Description", 

        "Customer_Email" 

    ).limit(20) 

) 

 
 
 

# ============================================================ 

# 8. WRITE TO SILVER DELTA TABLE 

# ============================================================ 

 
 

( 

    silver_df.write 

    .format("delta") 

    .mode("overwrite") 

    .option("overwriteSchema", "true") 

    .saveAsTable(TARGET_TABLE) 

) 

 
 
 

# ============================================================ 

# 9. VALIDATION 

# ============================================================ 

 
 

silver_count = spark.table(TARGET_TABLE).count() 

 
 

print("==============================================") 

print("Bronze -> Silver transformation completed") 

print("==============================================") 

print("Source Table :", SOURCE_TABLE) 

print("Target Table :", TARGET_TABLE) 

print("Silver Count :", silver_count) 

print("==============================================") 

 
 
 

# ============================================================ 

# 10. VERIFY FINAL SILVER DATA 

# ============================================================ 

 
 

display( 

    spark.sql(f""" 

        SELECT 

            Customer_ID, 

            Customer_Name, 

            Transaction_Date, 

            Transaction_Time, 

            Transaction_Location, 

            District, 

            Transaction_Amount, 

            Account_Balance, 

            Is_Fraud 

        FROM {TARGET_TABLE} 

        LIMIT 20 

    """) 

) 